# Sta-RU Video Dubbing — Edge-TTS variant

Same pipeline as the main notebook but uses **Microsoft Edge's neural TTS** (free, no API key, no voice cloning). Much faster (~30s per video instead of 6 min) and gives a stable, broadcast-grade voice.

Use this notebook when you don't need to preserve the original speaker's timbre and prefer a clean stock narrator.

## 1. GPU check (optional)

GPU is only used by Demucs. The TTS itself runs on CPU and is plenty fast.

In [ ]:
!nvidia-smi | head -20

## 2. Install dependencies

In [ ]:
!apt-get -qq install -y ffmpeg rubberband-cli
!pip install -q edge-tts nest-asyncio yt-dlp srt soundfile numpy scipy demucs deep-translator pyrubberband ipywidgets pandas

## 3. Clone the Sta-RU repo

In [ ]:
import os, sys
REPO_DIR = '/content/Sta-RU'
BRANCH = 'claude/compassionate-knuth-M2llL'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 -b $BRANCH https://github.com/lazy-money/sta-ru.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull --quiet
sys.path.insert(0, os.path.join(REPO_DIR, 'colab'))
print('Repo ready at', REPO_DIR, '(branch:', BRANCH, ')')

## 3b. Preflight: can this VM reach YouTube?

YouTube sometimes blocks Colab's datacenter IPs with *"Sign in to confirm you're not a bot"*. This cell catches that **up front** — using a public canary video ("Me at the zoo") — before you waste time pasting URLs and configuring everything.

- It uses the **same** player-client logic as the real pipeline (that's why it runs here, after cloning the repo, not literally after step 2): if the preflight passes, the batch will too.
- The pipeline cycles clients on its own — you don't have to do anything.
- **Cookies = last resort.** Only if this prints BLOCKED, run the cookies cell below.

In [ ]:
# Preflight: can we download from YouTube on this VM? Probe with a stable,
# public canary video ("Me at the zoo", the first video on YouTube).
from batch_dub import preflight_youtube

CANARY_URL = "https://www.youtube.com/watch?v=jNQXAC9IVRw"
print('Probing YouTube access with the canary:', CANARY_URL)
print('(tries several player clients; may take a few seconds)')
_res = preflight_youtube(CANARY_URL)
if _res['ok']:
    print("\nOK: YouTube reachable -- passed with client '%s'." % _res['client'])
    print('The VM is not blocked. Continue from step 4 (Mount Drive).')
elif _res['client'] is None:
    print('\nBLOCKED: YouTube bot check (cloud IP). Last error:')
    print('  ', _res['error'])
    print('   -> Re-run this cell (it sometimes passes on a retry).')
    print('   -> Change IP: Runtime > Disconnect and delete runtime, reconnect and retry.')
    print('   -> If it persists, run the COOKIES cell below (last resort).')
else:
    print('\nODD: the canary failed with an error that is NOT the bot check:')
    print('  ', _res['error'])
    print('   (Unusual for "Me at the zoo"; could be a VM network issue.)')

In [ ]:
# COOKIES (last resort) -- run this cell ONLY if the preflight above printed BLOCKED.
# 1) Install the "Get cookies.txt LOCALLY" extension (Chrome/Firefox), open
#    youtube.com logged in with a SECONDARY account, and export cookies.txt.
# 2) Run this cell and upload the file when prompted.
import batch_dub
from google.colab import files as _gfiles

_up = _gfiles.upload()
if _up:
    _name = next(iter(_up))
    _path = '/content/cookies.txt'
    with open(_path, 'wb') as _f:
        _f.write(_up[_name])
    batch_dub.YTDLP_COOKIES = _path
    print('Cookies loaded from %s -> %s' % (_name, _path))
    print('Now re-run the preflight above to confirm it passes.')
else:
    print('No file uploaded. (Run this cell only if the preflight failed.)')

## 4. Mount Google Drive (optional)

Run this **after** the preflight passes — mounting is slow, so there's no point doing it until you know the VM can reach YouTube.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Provide YouTube URLs

One per line. Use `2: https://...` to force a specific N#.

In [ ]:
import ipywidgets as W
from IPython.display import display

url_box = W.Textarea(
    value='',
    placeholder=('Paste one YouTube URL per line.\n'
                 'https://www.youtube.com/watch?v=...\n'
                 '\nOr force N#:\n'
                 '2: https://www.youtube.com/watch?v=...'),
    description='URLs:',
    layout=W.Layout(width='100%', height='180px'),
)
display(url_box)

## 6. Upload subtitle files

Format: `{N#}-{LANG}.srt` (e.g. `1-EN.srt`, `47-ES.srt`). Multi-select supported.

In [ ]:
from google.colab import files
import os, re

SRT_DIR = '/content/srts'
os.makedirs(SRT_DIR, exist_ok=True)
_srts = files.upload()
_pattern = re.compile(r'^(\d+)-([A-Z]{2})\.srt$')
loaded = []
for name, content in _srts.items():
    with open(os.path.join(SRT_DIR, name), 'wb') as f:
        f.write(content)
    m = _pattern.match(name)
    if m:
        loaded.append((int(m.group(1)), m.group(2)))
print(f'\nLoaded {len(loaded)} SRTs')
by_lang = {}
for n, lg in loaded:
    by_lang.setdefault(lg, []).append(n)
for lg, ns in by_lang.items():
    print(f'  {lg}: N# {sorted(ns)}')

## 7. Options

In [ ]:
import ipywidgets as W
import os
from IPython.display import display

_drive_mounted = os.path.ismount('/content/drive')
_default_out_mode = 'drive' if _drive_mounted else 'local'
_default_out_lang = 'EN'
_default_out_path = (f'/content/drive/MyDrive/Dubbing/{_default_out_lang}' if _drive_mounted
                     else f'/content/output/{_default_out_lang}')
from batch_dub_edge import DEFAULT_VOICES

LANG_OPTIONS = list(DEFAULT_VOICES.keys())

lang_w            = W.Dropdown(options=LANG_OPTIONS, value='EN', description='Target language:')
gender_w          = W.RadioButtons(options=[('Male', 'M'), ('Female', 'F')], value='M', description='Voice gender:')
voice_custom_w    = W.Text(value='', placeholder='Optional: override the default voice (e.g. en-US-BrianNeural)',
                           description='Custom voice:', layout=W.Layout(width='600px'))
pitch_w           = W.IntSlider(value=-5, min=-20, max=20, step=1, description='Pitch (Hz):')
range_w           = W.Text(value='all', description='Range (N#):', placeholder="'all', '1-10', '47'")
remove_voice_w    = W.Checkbox(value=True,  description='Remove original voice (Demucs)')
dynamic_dur_w     = W.Checkbox(value=True, description='Dynamic duration (stretch video to fit dubbing)')
skip_silent_w     = W.Checkbox(value=True,  description='Skip TTS where the original speaker is silent')
burn_subs_w       = W.Checkbox(value=False, description='Burn subtitles into the video')
translate_title_w = W.Checkbox(value=True, description='Translate title to target language')
hq_demucs_w       = W.Checkbox(value=False, description='Higher-quality vocal removal (mdx_extra) — slower')
allow_no_amb_w    = W.Checkbox(value=False, description='Continue without ambient if Demucs fails')
ambient_gain_w    = W.FloatSlider(value=3.5, min=1.0, max=8.0, step=0.25,
                                  description='Ambient gain:', readout_format='.2f')

out_mode_w = W.RadioButtons(
    options=[('Save to Google Drive', 'drive'), ('Save locally', 'local')],
    value=_default_out_mode, description='Output:',
)
out_path_w = W.Text(value=_default_out_path, description='Output dir:',
                    layout=W.Layout(width='600px'))
cache_path_w = W.Text(value='/tmp/sta-ru-cache', description='Cache dir:',
                      layout=W.Layout(width='600px'))

def _sync(_=None):
    lg = lang_w.value
    out_path_w.value = (f'/content/drive/MyDrive/Dubbing/{lg}' if out_mode_w.value == 'drive'
                        else f'/content/output/{lg}')
lang_w.observe(_sync, names='value')
out_mode_w.observe(_sync, names='value')

display(lang_w, gender_w, voice_custom_w, pitch_w, range_w,
        remove_voice_w, dynamic_dur_w, skip_silent_w, burn_subs_w,
        translate_title_w, hq_demucs_w, allow_no_amb_w, ambient_gain_w,
        out_mode_w, out_path_w, cache_path_w)
print()
print('Dynamic duration: voice stays natural; video stretches to match. ~3-5x slower per video.')
print('Skip silent: avoids dubbing over moments the speaker stayed quiet (Whisper hallucinations).')
print('Burn subtitles: bakes the SRT into the video (forces re-encode, slower).')
print('Cache dir: persists video/ambient per URL so other languages reuse them.')
print('Vocal removal: mdx_extra preserves workshop ambient better than the default htdemucs')
print('               but is ~3-4x slower per video. Toggle on for keepers, off for batches.')
print('Continue without ambient: by default a video is aborted if Demucs cannot')
print('                          extract the ambient stem (avoids producing dry dubs).')
print('                          Tick to override and let the video render anyway.')
print('Ambient gain: how loud the no-vocals stem sits under the TTS. 3.5 is the')
print('              default; raise for more room tone, lower if it competes with the voice.')
print()
print('Adjust the values above, then run the next cell.')
if not _drive_mounted:
    print()
    print('NOTE: Google Drive is not mounted (step 3 not run) — output defaulted to local.')
    print('      Mount Drive and re-run this cell if you want to save there instead.')

In [ ]:
# Lock in configuration
CONFIG = {
    'lang':                 lang_w.value,
    'gender':               gender_w.value,
    'voice':                voice_custom_w.value.strip() or None,
    'pitch_st':             pitch_w.value,
    'range_expr':           range_w.value,
    'remove_voice':         remove_voice_w.value,
    'dynamic_duration':     dynamic_dur_w.value,
    'skip_silent_segments': skip_silent_w.value,
    'burn_in_subs':         burn_subs_w.value,
    'translate_titles':     translate_title_w.value,
    'demucs_model':         'mdx_extra' if hq_demucs_w.value else 'htdemucs',
    'ambient_gain':         ambient_gain_w.value,
    'allow_no_ambient':     allow_no_amb_w.value,
    'output_dir':           out_path_w.value,
    'srt_dir':              SRT_DIR,
    'cache_root':           cache_path_w.value or None,
}

url_text = (url_box.value or '').strip()
if not url_text:
    raise RuntimeError('No URLs provided. Paste in the textarea.')
URL_SOURCE = [l for l in url_text.splitlines() if l.strip()]
print(f'URL source: textarea ({len(URL_SOURCE)} lines)')

from batch_dub_edge import resolve_voice
effective_voice = resolve_voice(CONFIG['lang'], CONFIG['gender'], CONFIG['voice'])
print(f'Effective voice: {effective_voice}')
print('\nConfiguration:')
for k, v in CONFIG.items():
    print(f'  {k:22} = {v}')

import os
if CONFIG['output_dir'].startswith('/content/drive/') and not os.path.ismount('/content/drive'):
    _fallback = f"/content/output/{CONFIG['lang']}"
    print()
    print(f'  [NOTICE] Drive is not mounted but output_dir points there.')
    print(f'           Falling back to local: {_fallback}')
    print(f'           Run step 3 before step 7 if you want Drive instead.')
    CONFIG['output_dir'] = _fallback

import batch_dub
batch_dub.AMBIENT_GAIN = CONFIG['ambient_gain']
print(f"  AMBIENT_GAIN set to {batch_dub.AMBIENT_GAIN} for this run")


## 8. Preview

In [ ]:
import pandas as pd
from batch_dub import load_urls, build_items, build_output_name

all_urls = load_urls(URL_SOURCE)
preview_urls = all_urls[:3]
preview_items = build_items(preview_urls,
                            translate_titles=CONFIG['translate_titles'],
                            target_lang=CONFIG['lang'].lower())
rows = []
for it in preview_items:
    rows.append({
        'N#': it.n,
        'Date': it.upload_date or '—',
        'Duration': f'{int(it.duration//60)}:{int(it.duration%60):02d}' if it.duration else '—',
        'Title': it.title or f'[fallback — {it.error}]',
        'Output filename': build_output_name(it, CONFIG['lang'], CONFIG['translate_titles']),
    })
pd.DataFrame(rows).style.set_properties(**{'text-align': 'left'}).hide(axis='index')

## 9. Run batch

In [ ]:
from batch_dub_edge import run_batch

results = run_batch(
    urls=URL_SOURCE,
    srt_dir=CONFIG['srt_dir'],
    output_dir=CONFIG['output_dir'],
    lang=CONFIG['lang'],
    gender=CONFIG['gender'],
    voice=CONFIG['voice'],
    pitch_st=CONFIG['pitch_st'],
    translate_titles=CONFIG['translate_titles'],
    remove_voice=CONFIG['remove_voice'],
    dynamic_duration=CONFIG['dynamic_duration'],
    skip_silent_segments=CONFIG['skip_silent_segments'],
    burn_in_subs=CONFIG['burn_in_subs'],
    cache_root=CONFIG['cache_root'],
    range_expr=CONFIG['range_expr'],
    demucs_model=CONFIG['demucs_model'],
    allow_no_ambient=CONFIG['allow_no_ambient'],
)

## 10. Download outputs (local mode only)

In [ ]:
from google.colab import files
from pathlib import Path
import os

drive_mounted = os.path.ismount('/content/drive')
n_done = n_drive = n_dl = 0
for it in results:
    if it.status != 'done' or not it.output_path:
        continue
    n_done += 1
    p = Path(it.output_path)
    if not p.exists():
        print(f'  [WARN] {p.name} marked done but file not found at {p}')
        continue
    on_drive = drive_mounted and str(p).startswith('/content/drive/')
    if on_drive:
        n_drive += 1
        print(f'  (skip) {p.name} — already in Drive')
        continue
    print(f'  Downloading {p.name}...')
    files.download(str(p))
    n_dl += 1
print(f'\n{n_done} done | {n_drive} in Drive | {n_dl} downloaded')

## Reset for next run

Wipes the cached Python modules from this session **and** the cloned repo on disk. Run this when you want the next iteration to pick up new code (after a `git pull` on the repo).

In [ ]:
# Wipe everything that gets stale between runs:
#   - cached batch_dub modules in this Python session
#   - cloned repo on disk (forces a fresh git clone next time you run cell 4)
# Run this when you want the next iteration to pick up new code.
import sys
for _m in list(sys.modules):
    if _m.startswith('batch_dub'):
        del sys.modules[_m]
print('Cleared cached batch_dub modules')

!rm -rf /content/Sta-RU
print('Removed /content/Sta-RU')

## Free disk: processing cache

Removes the cross-language cache (`/tmp/sta-ru-cache`) and per-video work dirs. Run this when you're done with a set of videos and don't plan to re-dub them in another language.

In [ ]:
import shutil, os
for p in ('/tmp/sta-ru-cache', '/tmp/sta-ru-edge', '/tmp/sta-ru-work', '/tmp/dbg', '/tmp/no_voice'):
    if os.path.exists(p):
        shutil.rmtree(p, ignore_errors=True)
        print(f'Removed {p}')
    else:
        print(f'(skip) {p} not present')
!df -h /tmp | tail -1

## Free disk: downloaded ML models

Removes the XTTS-v2 checkpoint (~2 GB) and the Demucs htdemucs weights (~80 MB). They'll be re-downloaded the next time you run the pipeline.

In [ ]:
import shutil, os
paths = [
    '/root/.local/share/tts',                     # Coqui XTTS-v2 model
    '/root/.cache/torch/hub/checkpoints',         # Demucs + other torch hub models
    '/root/.cache/huggingface',                   # HF cache (translation, etc)
]
for p in paths:
    if os.path.exists(p):
        shutil.rmtree(p, ignore_errors=True)
        print(f'Removed {p}')
    else:
        print(f'(skip) {p} not present')
!df -h / | tail -1